# Plan-and-Execute Agent
## Plan Once, Execute in Steps — Decomposition for Long Tasks
⏱ ~12 min

**Plan-and-Execute** is an agentic pattern where the LLM commits to a full step-by-step plan *before* touching any tool, then an executor loop works through each step in order, and a synthesizer assembles the final answer from all step results. This separation of concerns makes multi-step reasoning cheaper, more auditable, and easier to debug than interleaved ReAct loops.

By the end of this session you will understand *why* the plan/execute split exists, *how* every component is wired together with LangGraph, and *how* to extend the pattern with replanning, parallel execution, and structured tool use.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why separate planning from execution? |
| 2 | **Structured Output** — Forcing the LLM to return a typed `Plan` |
| 3 | **State Design** — `PlanState`, reducers, and the append-only results list |
| 4 | **Graph Assembly** — Planner → Executor loop → Synthesizer |
| 5 | **Running the Agent** — Tasks, inspection, and tracing |
| 6 | **Debugging** — Printing plans, step results, and prompt content |
| 7 | **Extensions** — Replanning, parallel steps, call counting |
| * | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ (a `.venv` with the requirements already installed)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> Wang, L., Xu, W., et al. (2023). *Plan-and-Solve Prompting: Improving Zero-Shot Chain-of-Thought Reasoning by Large Language Models.* ACL 2023. https://arxiv.org/abs/2305.04091  
> Yao, S., Zhao, J., et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* ICLR 2023. https://arxiv.org/abs/2210.03629  
> LangGraph documentation: https://langchain-ai.github.io/langgraph/  
> LangChain structured output: https://python.langchain.com/docs/how_to/structured_output/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain",
            "langchain-openai",
            "langgraph",
            "pydantic",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import operator
import os
from typing import Annotated, TypedDict

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Core imports ──────────────────────────────────────────────────────────────
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from pydantic import BaseModel, ValidationError

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(key) and key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED — add your key before running LLM cells.")
    print("  Local: echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'")

---

## Part 1 — What Is Plan-and-Execute and Why Does It Exist?

---

### The Problem With Interleaved Reasoning

The dominant agentic pattern before Plan-and-Execute was **ReAct** (Yao et al., 2023): the model alternates between *reasoning* (think) and *acting* (use a tool), observing the result after each action before deciding what to do next. This is powerful for open-ended, exploratory tasks but has three drawbacks:

1. **Token cost** — every think-act-observe cycle adds tokens. A 5-step task might take 10+ LLM calls.
2. **Drift** — the model can lose sight of the original goal over many iterations.
3. **Unpredictability** — the number of steps is unknown in advance, making latency hard to bound.

**Plan-and-Execute** (Wang et al., 2023) solves this by splitting the agent into two dedicated roles:
- A **Planner** that calls the LLM once to produce a complete step list
- An **Executor** that works through each step in order, using lighter LLM calls

---

### Three Approaches Compared

| Approach | LLM calls | Adaptability | Predictability | Best for |
|----------|-----------|--------------|----------------|---------|
| **Direct prompting** | 1 | None | High | Simple, single-step tasks |
| **ReAct** | N (interleaved) | High | Low | Open-ended, exploratory tasks |
| **Plan-Execute** | 1 plan + N executor | Medium | High | Structured, multi-step tasks |
| **BabyAGI / ReWOO** | 1 plan + parallel | Low | Very high | Long-horizon tasks with parallel steps |

---

### Key Vocabulary

| Term | Definition |
|------|------------|
| **Planner** | The LLM node that generates a typed list of steps upfront |
| **Step** | A single, concrete, actionable instruction from the plan |
| **Executor** | The loop node that processes one step per graph tick |
| **Synthesizer** | The node that combines all step results into a final answer |
| **Structured output** | `with_structured_output(Plan)` — forces the LLM to return a Pydantic object |
| **Reducer** | `Annotated[list[str], operator.add]` — tells LangGraph to append rather than replace |
| **Conditional edge** | A router function that decides which node runs next (loop vs exit) |

### Plan-and-Execute Architecture

```
USER QUERY
    |
    v
+-----------------------------------+
|  PLANNER  (one LLM call)          |  with_structured_output(Plan)
|  "Break this into 3-5 steps"      |  --> Plan(steps=[Step(...), ...])
+----------------+------------------+
                 |
                 v
       steps = [step1, step2, step3]
                 |
     +-----------v-----------+
     |  EXECUTOR             |<--------------+
     |  runs steps[0]        |               |
     |  appends result       |               |  still steps remaining?
     |  pops step from list  |               |
     +-----------+-----------+               |
                 |                           |
      steps empty? --- NO -------------------+
                 |
                YES
                 |
     +-----------v-----------+
     |  SYNTHESIZER          |  combines results[0..N]
     |  "Combine N results"  |  into one coherent answer
     +-----------+-----------+
                 |
                 v
           FINAL ANSWER
```

> **Why `operator.add` on results?** LangGraph's default state merge *replaces* fields. We want each executor call to *append* one result to the list — not overwrite. The `Annotated[list[str], operator.add]` type annotation tells LangGraph to use the `+` reducer instead of replacement.

---

## Part 2 — Structured Output: Forcing a Typed Plan

---

The most important primitive in Plan-and-Execute is `with_structured_output`. Instead of returning free text that you parse with regex or string splitting, the LLM is instructed to return a JSON object validated against a Pydantic schema.

### How It Works

```
llm.with_structured_output(Plan)
    |
    v  Wraps the LLM call with a tool definition derived from the Plan schema
    |
    v  The model fills in the schema as a JSON tool call
    |
    v  Pydantic validates the output — raises ValidationError if malformed
    |
    v  Returns a Plan object with .steps: list[Step]
```

### Planner Design Patterns

| Pattern | Description | When to use |
|---------|-------------|-------------|
| **Fixed schema** | Pydantic model with `steps: list[Step]` | Most cases — predictable, type-safe |
| **Steps with tools** | Each step includes a `tool_name` field | When steps map to specific tools |
| **Dependency graph** | Steps include `depends_on: list[int]` | When steps can run in parallel |
| **Conditional steps** | Steps include `condition: str` | When plan branches on outcomes |
| **Replanning prompt** | Re-invoke planner with current results | When executor hits a dead end |

In [ ]:
# ─── 2-A: Define the Plan schema ─────────────────────────────────────────────
# Pydantic models are the source of truth for structured output.
# The LLM sees these field names and docstrings as part of the tool definition.


class Step(BaseModel):
    """A single, concrete, actionable step in the plan."""

    step: str


class Plan(BaseModel):
    """A structured plan decomposing a task into sequential steps."""

    steps: list[Step]


# with_structured_output wires the schema to the LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
planner_llm = llm.with_structured_output(Plan)

print("Plan JSON schema:")
import json
print(json.dumps(Plan.model_json_schema(), indent=2))

In [ ]:
# ─── 2-B: Call the planner and inspect the typed output ──────────────────────
# Key insight: the return value is a Plan object, not a string.
# No parsing, no regex — Pydantic validation is automatic.

PLANNER_SYSTEM = (
    "You are a task planner. Break the user's task into 3-5 clear, sequential steps. "
    "Each step should be a concrete action that moves toward the goal. "
    "Return ONLY the structured plan, no preamble."
)

test_task = "Write a launch plan for a developer tool startup."

plan = planner_llm.invoke(
    [SystemMessage(PLANNER_SYSTEM), HumanMessage(test_task)]
)

print(f"Type returned: {type(plan)}")
print(f"Number of steps: {len(plan.steps)}")
print()
for i, s in enumerate(plan.steps, 1):
    print(f"  Step {i}: {s.step}")

In [ ]:
# ─── 2-C: What happens when the output is malformed? ─────────────────────────
# with_structured_output retries internally for minor deviations.
# For a hard failure demonstration, we manually validate bad data.

# Missing required field — Pydantic raises ValidationError
try:
    bad_step = Step()  # 'step' field is required
except ValidationError as e:
    print("ValidationError caught (expected):")
    print(e)

print()

# A valid step for comparison
good_step = Step(step="Research the target market and define the ICP")
print(f"Valid step: {good_step}")
print(f"Field access: plan.steps[0].step = '{good_step.step[:50]}'")

---

## Part 3 — State Design: PlanState and Reducers

---

LangGraph manages agent state as a `TypedDict`. Every node receives the current state and returns a *partial update* — only the fields it changes. LangGraph merges updates back into the shared state.

### The Default Merge Behavior

By default, a node returning `{"results": ["step 1 done"]}` would *replace* the existing results list. That would lose every prior result.

### The Append-Only Reducer

```python
from typing import Annotated
import operator

class PlanState(TypedDict):
    task: str                                    # input, never changed
    steps: list[str]                             # shrinks each executor tick
    results: Annotated[list[str], operator.add]  # GROWS each tick (appends)
    final_answer: str                            # set once by synthesizer
```

`Annotated[list[str], operator.add]` tells LangGraph: when merging `results`, call `operator.add(existing, new)` — list concatenation. So returning `{"results": ["one result"]}` appends rather than replaces.

### State Lifecycle

```
After planner:     steps=[s1,s2,s3]  results=[]          final_answer=""
After executor 1:  steps=[s2,s3]     results=[r1]        final_answer=""
After executor 2:  steps=[s3]        results=[r1,r2]     final_answer=""
After executor 3:  steps=[]          results=[r1,r2,r3]  final_answer=""
After synthesizer: steps=[]          results=[r1,r2,r3]  final_answer="..."
```

In [ ]:
# ─── 3-A: Define PlanState ────────────────────────────────────────────────────


class PlanState(TypedDict):
    task: str
    steps: list[str]  # remaining steps; executor pops from front
    results: Annotated[list[str], operator.add]  # append-only reducer
    final_answer: str


# Demonstrate the reducer behavior manually
print("operator.add on lists (list concatenation):")
existing = ["result 1", "result 2"]
new_result = ["result 3"]
merged = operator.add(existing, new_result)
print(f"  {existing} + {new_result} = {merged}")

print()
print("Default dict.update behavior (replace — NOT what we want):")
state = {"results": ["result 1", "result 2"]}
update = {"results": ["result 3"]}
state.update(update)  # this replaces!
print(f"  After dict.update: {state['results']}  <- WRONG (lost results 1 and 2)")

In [ ]:
# ─── 3-B: Visualise state transitions ────────────────────────────────────────
# Simulate what happens to state at each node (without LLM calls).

import copy

initial_state: PlanState = {
    "task": "Build a landing page for a SaaS product",
    "steps": ["Research competitors", "Write copy", "Design wireframe"],
    "results": [],
    "final_answer": "",
}


def simulate_executor_tick(state: PlanState, fake_result: str):
    """Simulate one executor tick without an LLM call."""
    new_state = copy.deepcopy(state)
    completed_step = new_state["steps"].pop(0)
    new_state["results"] = operator.add(new_state["results"], [fake_result])
    return new_state, completed_step


state = initial_state
fake_results = [
    "Analyzed 5 competitor landing pages",
    "Drafted headline and value prop",
    "Created 3-section wireframe",
]

print("STATE TRACE (simulated, no LLM calls):\n")
print(f"Initial:  steps={state['steps']}  results={state['results']}")
print()

for fake in fake_results:
    state, done = simulate_executor_tick(state, fake)
    print(f"Completed: '{done}'")
    print(f"  steps={state['steps']}")
    print(f"  results={state['results']}")
    print()

---

## Part 4 — Graph Assembly: Wiring the Nodes

---

A LangGraph graph has three kinds of connections:

| Connection type | Code | Effect |
|----------------|------|--------|
| `add_edge(A, B)` | Unconditional | A always goes to B |
| `add_conditional_edges(A, fn, map)` | Conditional | fn(state) returns a key, map routes to a node |
| `set_entry_point(node)` | Entry | First node to run |
| `add_edge(X, END)` | Terminal | X exits the graph |

The executor uses a **conditional self-loop**: after each step, if `steps` is non-empty it routes back to itself; once empty it exits to the synthesizer.

In [ ]:
# ─── 4-A: Define node functions ───────────────────────────────────────────────

EXECUTOR_SYSTEM = (
    "You are a task executor. Complete the given step thoroughly in 2-3 sentences. "
    "Reference prior results where relevant."
)

SYNTHESIZER_SYSTEM = (
    "You are a synthesizer. Combine all completed step results into a coherent, "
    "final response to the original task. Be concise and well-structured."
)


def planner(state: PlanState) -> dict:
    """Call the LLM once to generate a full structured plan."""
    plan = planner_llm.invoke(
        [SystemMessage(PLANNER_SYSTEM), HumanMessage(state["task"])]
    )
    steps = [s.step for s in plan.steps]
    print(f"  [planner] {len(steps)} steps: {steps}")
    return {"steps": steps, "results": []}


def executor(state: PlanState) -> dict:
    """Execute the first remaining step and append its result."""
    step = state["steps"][0]
    prior = "\n".join(f"Step result: {r}" for r in state["results"])
    prompt = (
        f"Task context: {state['task']}\n\n"
        f"Prior results:\n{prior}\n\n"
        f"Execute this step: {step}"
    )
    result = llm.invoke([SystemMessage(EXECUTOR_SYSTEM), HumanMessage(prompt)])
    print(f"  [executor] done: {step[:55]}")
    # Return remaining steps (front-popped) and the new result
    return {"steps": state["steps"][1:], "results": [result.content]}


def synthesizer(state: PlanState) -> dict:
    """Combine all step results into the final answer."""
    combined = "\n".join(f"Step {i + 1}: {r}" for i, r in enumerate(state["results"]))
    prompt = f"Task: {state['task']}\n\nCompleted steps:\n{combined}"
    answer = llm.invoke([SystemMessage(SYNTHESIZER_SYSTEM), HumanMessage(prompt)])
    print(f"  [synthesizer] assembled final answer")
    return {"final_answer": answer.content}


def should_continue(state: PlanState) -> str:
    """Router: loop back to executor while steps remain; else exit to synthesizer."""
    return "executor" if state["steps"] else "synthesizer"


print("Node functions defined.")

In [ ]:
# ─── 4-B: Assemble and compile the graph ──────────────────────────────────────

graph = StateGraph(PlanState)

graph.add_node("planner", planner)
graph.add_node("executor", executor)
graph.add_node("synthesizer", synthesizer)

graph.set_entry_point("planner")
graph.add_edge("planner", "executor")  # always run executor first after planning
graph.add_conditional_edges(
    "executor",
    should_continue,
    {"executor": "executor", "synthesizer": "synthesizer"},
)
graph.add_edge("synthesizer", END)

app = graph.compile()
print("Graph compiled.")
print()
print("Topology:")
print("  START --> planner --> executor --+")
print("                           ^       | (steps remaining)")
print("                           +-------+")
print("                           |")
print("                           | (steps empty)")
print("                           v")
print("                      synthesizer --> END")

---

## Part 5 — Running the Agent

---

Every LangGraph app is invoked with `app.invoke(initial_state)`. The initial state must provide all required TypedDict fields. Fields managed by nodes (`steps`, `results`, `final_answer`) should be seeded as empty.

In [ ]:
# ─── 5-A: Run a single task ───────────────────────────────────────────────────

TASK = "Write a 3-step launch plan for a developer tool startup."

print(f"TASK: {TASK}")
print("-" * 60)

result = app.invoke(
    {"task": TASK, "steps": [], "results": [], "final_answer": ""}
)

print()
print(f"Steps executed: {len(result['results'])}")
print()
print("FINAL ANSWER:")
print(result["final_answer"])

In [ ]:
# ─── 5-B: Inspect intermediate step results ───────────────────────────────────
# result['results'] holds each step's output — key for debugging and auditing.

print("STEP-BY-STEP RESULTS:\n")
for i, step_result in enumerate(result["results"], 1):
    print(f"Step {i}:")
    print(f"  {step_result[:200]}")
    if len(step_result) > 200:
        print(f"  ... [{len(step_result) - 200} more chars]")
    print()

In [ ]:
# ─── 5-C: Run multiple tasks ──────────────────────────────────────────────────

SAMPLE_TASKS = [
    "Outline how to learn LangGraph in 4 weeks as a Python developer.",
    "Create a plan for migrating a monolith to microservices.",
]

for task in SAMPLE_TASKS:
    print(f"\nTASK: {task}")
    print("-" * 60)
    r = app.invoke({"task": task, "steps": [], "results": [], "final_answer": ""})
    print(f"\nSteps: {len(r['results'])}")
    print(f"Answer preview: {r['final_answer'][:300]}")
    print()

---

## Part 6 — Debugging the Agent

---

### Common Plan-Execute Failure Modes

| Failure | Symptom | Root cause | Fix |
|---------|---------|-----------|-----|
| **Steps too vague** | Executor produces generic output | Planner prompt too loose | Add specificity requirement to planner system prompt |
| **Too many steps** | 10+ trivial steps for a simple task | No step count constraint | Set `3-5 clear steps` in planner prompt |
| **Context loss** | Executor ignores prior results | Prior results not in prompt | Inject `state['results']` into executor prompt |
| **Hallucinated steps** | Steps reference facts not in task | Temperature too high | Lower planner temperature to 0.0-0.3 |
| **Synthesizer ignores steps** | Final answer omits step work | Synthesizer prompt too weak | Explicitly list all step results in synthesizer prompt |
| **Infinite loop** | `should_continue` always returns executor | Bug in step-popping logic | Verify `state['steps'][1:]` is returned, not `state['steps']` |

In [ ]:
# ─── 6-A: Diagnostic — print the plan before execution ────────────────────────
# Always inspect the plan first. Bad plans produce bad results regardless
# of executor quality.

debug_task = "Research and summarize the pros and cons of GraphQL vs REST."

debug_plan = planner_llm.invoke(
    [SystemMessage(PLANNER_SYSTEM), HumanMessage(debug_task)]
)

print(f"Task: {debug_task}")
print(f"Steps generated: {len(debug_plan.steps)}")
print()
for i, s in enumerate(debug_plan.steps, 1):
    print(f"  [{i}] {s.step}")

print()
print("Diagnostic questions:")
print("  - Are the steps concrete and actionable?")
print("  - Do they cover the full task, or is something missing?")
print("  - Would a human following these steps produce a good answer?")

In [ ]:
# ─── 6-B: Diagnostic — print the exact prompt the executor sees ───────────────
# The most common issue: executor doesn't have enough context.

from langchain_core.callbacks import BaseCallbackHandler


class PromptInspector(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        print("=== PROMPT SENT TO LLM ===")
        for prompt_text in prompts:
            print(prompt_text[:800])
            if len(prompt_text) > 800:
                print(f"... [{len(prompt_text) - 800} more chars]")
        print("=" * 40)


# One-shot manual executor call for debugging
debug_step = "List 3 key advantages of GraphQL over REST"
debug_prior = ["GraphQL was developed by Facebook in 2012 and open-sourced in 2015."]

prior_text = "\n".join(f"Step result: {r}" for r in debug_prior)
debug_prompt = (
    f"Task context: {debug_task}\n\n"
    f"Prior results:\n{prior_text}\n\n"
    f"Execute this step: {debug_step}"
)

debug_llm = ChatOpenAI(
    model="gpt-4o-mini", temperature=0.3, callbacks=[PromptInspector()]
)
_ = debug_llm.invoke([SystemMessage(EXECUTOR_SYSTEM), HumanMessage(debug_prompt)])

---

## Part 7 — Extensions: Replanning and Call Counting

---

### Replanning Strategies

| Strategy | Trigger | Implementation |
|----------|---------|----------------|
| **Error-triggered replan** | Executor raises an exception | Catch exception → update state → route to planner |
| **Quality-triggered replan** | Synthesizer deems answer insufficient | Add `critic` node that checks quality score |
| **Iterative replan** | Fixed N rounds of plan→execute→replan | Loop counter in state |
| **ReWOO-style** | Plan ALL tool calls upfront with evidence placeholders | Separate plan from tool execution |

Extended graph with replanning:

```
planner --> executor (loop) --> synthesizer --> critic
   ^                                              |
   |                          answer ok? -- YES --> END
   |                                    |
   +------------------ NO (replan) <----+
```

In [ ]:
# ─── 7-A: Critic node and replan routing ──────────────────────────────────────
# Extends the base graph with a quality check after synthesis.

CRITIC_SYSTEM = (
    "You are a quality critic. Evaluate the final answer. "
    "If it is complete and specific, respond with exactly: ACCEPT\n"
    "If it is too vague or missing key points, respond with: REVISE: <one-line feedback>"
)


class PlanStateWithReplan(TypedDict):
    task: str
    steps: list[str]
    results: Annotated[list[str], operator.add]
    final_answer: str
    replan_count: int   # guard against infinite replan loops
    critic_feedback: str


def critic(state: PlanStateWithReplan) -> dict:
    verdict = llm.invoke(
        [
            SystemMessage(CRITIC_SYSTEM),
            HumanMessage(f"Task: {state['task']}\n\nAnswer: {state['final_answer']}"),
        ]
    )
    feedback = verdict.content.strip()
    print(f"  [critic] {feedback[:80]}")
    return {"critic_feedback": feedback}


def should_replan(state: PlanStateWithReplan) -> str:
    if state["replan_count"] >= 2:
        return "end"  # hard cap — avoid infinite loops
    if state["critic_feedback"].upper().startswith("ACCEPT"):
        return "end"
    return "replan"


def replan_planner(state: PlanStateWithReplan) -> dict:
    task_with_feedback = (
        f"{state['task']}\n\nCritic feedback: {state['critic_feedback']}\n"
        f"Prior answer to improve: {state['final_answer']}"
    )
    plan = planner_llm.invoke([SystemMessage(PLANNER_SYSTEM), HumanMessage(task_with_feedback)])
    steps = [s.step for s in plan.steps]
    attempt = state["replan_count"] + 2
    print(f"  [replan] {len(steps)} steps (attempt {attempt})")
    return {"steps": steps, "results": [], "replan_count": state["replan_count"] + 1, "final_answer": ""}


# Build the extended graph
rg = StateGraph(PlanStateWithReplan)
rg.add_node("planner", replan_planner)
rg.add_node("executor", executor)
rg.add_node("synthesizer", synthesizer)
rg.add_node("critic", critic)
rg.set_entry_point("planner")
rg.add_edge("planner", "executor")
rg.add_conditional_edges("executor", should_continue, {"executor": "executor", "synthesizer": "synthesizer"})
rg.add_edge("synthesizer", "critic")
rg.add_conditional_edges("critic", should_replan, {"end": END, "replan": "planner"})
replan_app = rg.compile()
print("Replan graph compiled.")

In [ ]:
# ─── 7-B: Count LLM calls for Plan-Execute ────────────────────────────────────
# Concrete measurement of the cost advantage over ReAct.

call_log = {"count": 0}


class CallCounter(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        call_log["count"] += 1


counting_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3, callbacks=[CallCounter()])
counting_planner_llm = counting_llm.with_structured_output(Plan)


def counting_planner(state):
    plan = counting_planner_llm.invoke([SystemMessage(PLANNER_SYSTEM), HumanMessage(state["task"])])
    return {"steps": [s.step for s in plan.steps], "results": []}


def counting_executor(state):
    step = state["steps"][0]
    prior = "\n".join(f"Result: {r}" for r in state["results"])
    result = counting_llm.invoke(
        [SystemMessage(EXECUTOR_SYSTEM), HumanMessage(f"Task: {state['task']}\n{prior}\nStep: {step}")]
    )
    return {"steps": state["steps"][1:], "results": [result.content]}


def counting_synthesizer(state):
    combined = "\n".join(f"Step {i + 1}: {r}" for i, r in enumerate(state["results"]))
    answer = counting_llm.invoke(
        [SystemMessage(SYNTHESIZER_SYSTEM), HumanMessage(f"Task: {state['task']}\n{combined}")]
    )
    return {"final_answer": answer.content}


cg = StateGraph(PlanState)
cg.add_node("planner", counting_planner)
cg.add_node("executor", counting_executor)
cg.add_node("synthesizer", counting_synthesizer)
cg.set_entry_point("planner")
cg.add_edge("planner", "executor")
cg.add_conditional_edges("executor", should_continue, {"executor": "executor", "synthesizer": "synthesizer"})
cg.add_edge("synthesizer", END)
counting_app = cg.compile()

call_log["count"] = 0
count_task = "Create a 3-step developer onboarding plan."
counting_app.invoke({"task": count_task, "steps": [], "results": [], "final_answer": ""})

n_steps = call_log["count"] - 2  # subtract planner + synthesizer
print(f"Task: '{count_task}'")
print(f"LLM calls (Plan-Execute): {call_log['count']}")
print(f"  1 planner + {n_steps} executor + 1 synthesizer")
print(f"  ReAct equivalent: ~{n_steps * 3} calls (think+act+observe per step)")
print(f"  Estimated savings: ~{n_steps * 3 - call_log['count']} fewer LLM calls")

---

## Exercises

---

### Exercise 1 — Inspect Structured Output

Call `planner_llm.invoke(...)` with a task of your choice. Print the raw `plan` object. What Python type is it? Access `plan.steps[0].step` directly — how does this compare to parsing a string response with regex?

**Bonus:** Add a required field `rationale: str` to the `Step` model. Re-run. What does the planner now return? What happens if the field is missing?

### Exercise 2 — Add a Re-planning Loop

Extend `PlanState` with a `replan_count: int` field. After synthesis, add a `critic` node that checks if `final_answer` is shorter than 100 characters (too brief). If so, route back to the planner with feedback: "The answer is too brief — expand with specific details." Cap replanning at 2 attempts.

### Exercise 3 — Count LLM Calls vs a Simple Chain

Using the `CallCounter` callback from Part 7-B, measure how many LLM calls Plan-Execute makes for a 4-step task. Then write a direct single-call chain (one LLM call with a chain-of-thought prompt). Compare: when does the extra structure of Plan-Execute pay off in output quality, and when is it wasteful?

### Exercise 4 — Parallel Step Execution

Change the executor to use `asyncio.gather` to run all steps concurrently. When is this safe (steps are independent) versus unsafe (step 2 needs the result of step 1)? Add a `depends_on: list[int]` field to `Step` and implement a check before parallelising.

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
my_task = "TODO: replace with your own task"

# TODO: call planner_llm.invoke([SystemMessage(PLANNER_SYSTEM), HumanMessage(my_task)])
# TODO: print type(result)
# TODO: print result.steps[0].step
# TODO: confirm you can access the field without any string parsing

# BONUS: add rationale field and re-run
# class StepWithRationale(BaseModel):
#     step: str
#     rationale: str
#
# class PlanWithRationale(BaseModel):
#     steps: list[StepWithRationale]
#
# rationale_planner = llm.with_structured_output(PlanWithRationale)

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────

# TODO: extend PlanState with replan_count: int
# TODO: write a critic node that checks len(final_answer) < 100
# TODO: write should_replan() returning 'end' or 'replan'
# TODO: wire critic into the graph after synthesizer
# TODO: invoke with replan_count=0 in initial state

print("Exercise 2: implement a replanning loop above")

In [ ]:
# ── Exercise 3 Starter ────────────────────────────────────────────────────────

# TODO: use CallCounter from Part 7-B to count Plan-Execute calls
# TODO: write a single-call chain: llm.invoke([SystemMessage(cot_prompt), HumanMessage(task)])
# TODO: count calls for both and compare output quality side-by-side
# NOTE: for simple 2-3 step tasks, a direct call may match quality at 1/5 the cost

print("Exercise 3: measure and compare LLM call counts above")

In [ ]:
# ── Exercise 4 Starter ────────────────────────────────────────────────────────

# TODO: write async_executor(state) using llm.ainvoke
# TODO: collect all step results with asyncio.gather(*tasks)
# TODO: add depends_on: list[int] to Step model
# TODO: implement check: only parallelize steps with empty depends_on

# HINT:
# import asyncio
# async def async_executor(state):
#     tasks = [llm.ainvoke([SystemMessage(EXECUTOR_SYSTEM), HumanMessage(step)])
#              for step in state['steps']]
#     responses = await asyncio.gather(*tasks)
#     return {'steps': [], 'results': [r.content for r in responses]}

print("Exercise 4: implement parallel execution above")

---

## What's Next?

You now have a complete foundation in Plan-and-Execute agents. Here's where to go deeper:

### Try closely related patterns
- **Example 4 — Basic ReAct Agent** (`../4-basic-react-agent/`): Compare with interleaved think-act-observe. Use ReAct when the task is open-ended and the model must adapt mid-stream based on tool output.
- **Example 37 — ReWOO Agent** (`../37-rewoo-agent/`): Emit all tool calls as a dependency-annotated plan upfront, execute in bulk. The next evolution of Plan-Execute for tool-heavy workflows.
- **Example 18 — Self-Reflecting Agent** (`../18-self-reflecting-agent/`): Add a structured critique/revise loop after synthesis — the same critic pattern introduced in Part 7.

### Improve the planner
- **Richer schemas**: Add `tool_name: str | None`, `depends_on: list[int]`, or `expected_output: str` to `Step` to give the executor more structure.
- **Temperature sweep**: Try planner temperatures from 0.0 to 0.8. At 0.0 the plan is deterministic; at 0.8 you get more creative, varied steps.
- **Few-shot examples**: Inject 2-3 example plans as part of the planner system prompt. This dramatically improves step quality on domain-specific tasks.

### Go to production
- **LangSmith tracing**: Set `LANGCHAIN_TRACING_V2=true` and `LANGCHAIN_API_KEY=...` to get a visual trace of every node and LLM call in a dashboard.
- **Streaming**: Replace `app.invoke` with `app.stream` to yield node outputs as they complete — useful for showing progress in a UI.
- **Human-in-the-loop**: Use `interrupt_before=["executor"]` to pause after planning and let a human approve the step list before execution starts.

### Further reading
- Wang et al. (2023). *Plan-and-Solve Prompting.* ACL 2023. https://arxiv.org/abs/2305.04091
- Yao et al. (2023). *ReAct: Synergizing Reasoning and Acting.* ICLR 2023. https://arxiv.org/abs/2210.03629
- Xu et al. (2023). *ReWOO: Decoupling Reasoning from Observations.* https://arxiv.org/abs/2305.18323
- LangGraph conceptual guide: https://langchain-ai.github.io/langgraph/concepts/
- LangGraph human-in-the-loop how-to: https://langchain-ai.github.io/langgraph/how-tos/human_in_the_loop/

---

*Workshop complete. Next step: try example 37 (ReWOO) to see what happens when you take the plan-execute idea further and execute all tool calls in parallel.*

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself. These are sample solutions, not the only correct answers.

### Exercise 1 — Inspect Structured Output

**Expected type:** `Plan` — a Pydantic `BaseModel` instance, not a string.

**Key insight:** Accessing `plan.steps[0].step` is immediate and type-safe. With a raw string response you would need to parse JSON or split on newlines and strip numbering — fragile code that breaks when the model changes its formatting.

**With `rationale` field:** The planner will include a one-sentence explanation for each step. If the model omits `rationale` for any step, Pydantic raises a `ValidationError` at the boundary — catching incomplete output before it silently propagates downstream.

**What good output looks like:**
```
Type: <class 'Plan'>
Fields: ['steps']
  [1] Define target audience and ICP
  [2] Build an MVP with core workflow automation features
  [3] Launch on Product Hunt and engage developer communities
```

In [ ]:
# Answer Key — Exercise 1

sample_task = "Write a plan for launching an open-source Python library."
sample_plan = planner_llm.invoke([SystemMessage(PLANNER_SYSTEM), HumanMessage(sample_task)])

print(f"Type: {type(sample_plan)}")
print(f"Fields: {list(sample_plan.model_fields.keys())}")
print()
for i, s in enumerate(sample_plan.steps, 1):
    print(f"  [{i}] {s.step}")

print()
print("BONUS — Step schema with rationale:")


class StepWithRationale(BaseModel):
    step: str
    rationale: str


class PlanWithRationale(BaseModel):
    steps: list[StepWithRationale]


rationale_llm = llm.with_structured_output(PlanWithRationale)
rplan = rationale_llm.invoke([SystemMessage(PLANNER_SYSTEM), HumanMessage(sample_task)])

for i, s in enumerate(rplan.steps, 1):
    print(f"  [{i}] {s.step}")
    print(f"       why: {s.rationale}")

### Exercise 2 — Add a Re-planning Loop

**Key design decisions:**
1. Add `replan_count: int` to state and initialise to `0` in `app.invoke`.
2. The critic node returns a string verdict; the string prefix drives the router.
3. `should_replan` returns `"end"` on ACCEPT or when `replan_count >= 2` (hard cap); otherwise `"replan"`.
4. The replan-planner receives both the original task AND the critic's feedback so the new plan directly addresses the gap.

**What to look for:** The first answer is often accepted. If your critic is triggering replanning every time, the threshold is too strict — loosen the acceptance condition.

**Full implementation is shown in Part 7-A** — the `PlanStateWithReplan`, `critic`, `should_replan`, and `replan_planner` functions above are the complete answer.

### Exercise 3 — Count LLM Calls vs a Simple Chain

**Expected call counts for a 4-step task:**

| Pattern | LLM calls | Formula |
|---------|-----------|--------|
| Direct chain (CoT prompt) | 1 | Always |
| Plan-Execute (4 steps) | 6 | 1 + N + 1 |
| ReAct (4 iterations) | ~9 | 2N + 1 |

**When Plan-Execute wins over direct:** Tasks where breaking the work into concrete sub-tasks genuinely improves quality — e.g., research tasks, structured reports, migration plans. The planner's decomposition forces the executor to be systematic.

**When direct wins:** Simple conversational tasks, summaries, or tasks where a single well-crafted prompt is sufficient. Adding 6 LLM calls for a task that a single call handles well is wasteful.

### Exercise 4 — Parallel Step Execution

**When parallel is safe:** Steps with no data dependency — e.g., `["Research audience", "Research competitors", "Research pricing"]` can all run concurrently because none needs the output of the others.

**When parallel is unsafe:** Steps like `["Write outline", "Expand outline into sections"]` — step 2 needs step 1's output as its input context.

**Dependency check implementation:**

```python
class StepWithDeps(BaseModel):
    step: str
    depends_on: list[int] = []  # indices of steps this step depends on

def can_parallelize(steps_with_deps: list[StepWithDeps]) -> bool:
    return all(len(s.depends_on) == 0 for s in steps_with_deps)
```

**Async implementation:**

```python
import asyncio

async def async_parallel_executor(state: PlanState) -> dict:
    tasks = [
        llm.ainvoke([SystemMessage(EXECUTOR_SYSTEM), HumanMessage(step)])
        for step in state['steps']
    ]
    responses = await asyncio.gather(*tasks)
    return {'steps': [], 'results': [r.content for r in responses]}
```

**Note:** When steps run in parallel, the executor does not have access to prior step results when forming its prompt. For fully independent steps this is acceptable; for dependent steps the sequential executor is required.